# Practical Session 3: Linear Programming for Production and Sales Planning

The original notebook in this slot was outdated. It has been rewritten so that it now
matches the current production-planning practical exactly and explains each modeling
step in a student-friendly way.


In [1]:
# Import NumPy for vector and matrix handling.
import numpy as np
# Import pandas so plans and scenario summaries can be displayed as tables.
import pandas as pd
# Import cvxpy to formulate the production-planning LPs.
import cvxpy as cp

# Format floating-point output compactly.
np.set_printoptions(precision=4, suppress=True)
# Choose a stable LP solver if it is available.
SOLVER = "CLARABEL" if "CLARABEL" in cp.installed_solvers() else "SCS"
# Fix the random seed for the automatic experiment framework.
rng = np.random.default_rng(11)

# Store product names once so tables remain readable.
products = ["A", "B", "C"]
# Profit per produced and sold unit.
profit = np.array([20, 25, 18], dtype=float)
# Machine-1 processing time per unit.
machine_1 = np.array([2, 3, 1], dtype=float)
# Machine-2 processing time per unit.
machine_2 = np.array([1, 1, 2], dtype=float)
# Capacity of the two machines.
capacity = np.array([100, 80], dtype=float)
# Maximum demand for each product.
demand_max = np.array([40, 30, 50], dtype=float)


## Tasks 1 and 2: Baseline production planning model

The baseline model maximizes total profit subject to machine capacities and demand
upper bounds.


In [2]:
def solve_single_period(profit, machine_1, machine_2, capacity, demand_max):
    # Create one nonnegative production variable for each product.
    # TODO: Create one nonnegative production variable per product.
    x = ...
    # Add the two machine-capacity constraints and the demand limits.
    constraints = [
        # TODO: Add the machine-capacity and demand constraints.
        ...,
    ]
    # Maximize total profit over the chosen production plan.
    problem = cp.Problem(cp.Maximize(profit @ x), constraints)
    problem.solve(solver=SOLVER)
    return problem, x


# Solve the baseline model from the script.
baseline_problem, baseline_x = solve_single_period(profit, machine_1, machine_2, capacity, demand_max)
baseline_plan = baseline_x.value

# Compute machine usage so unused capacity can be reported explicitly.
used_capacity = np.array([machine_1 @ baseline_plan, machine_2 @ baseline_plan])
summary = {
    "production_quantities": dict(zip(products, baseline_plan.round(4))),
    "total_profit": float(baseline_problem.value),
    "used_capacity": {"machine_1": used_capacity[0], "machine_2": used_capacity[1]},
    "unused_capacity": {
        "machine_1": capacity[0] - used_capacity[0],
        "machine_2": capacity[1] - used_capacity[1],
    },
    "active_demand_bounds": {
        product: bool(np.isclose(baseline_plan[idx], demand_max[idx], atol=1e-6))
        for idx, product in enumerate(products)
    },
    "constraints_satisfied": {
        "machine_1": bool(used_capacity[0] <= capacity[0] + 1e-6),
        "machine_2": bool(used_capacity[1] <= capacity[1] + 1e-6),
        "demand": bool(np.all(baseline_plan <= demand_max + 1e-6)),
    },
}
print("Status:", baseline_problem.status)
display(pd.DataFrame([summary]))


Status: optimal


,production_quantities,total_profit,used_capacity,unused_capacity,active_demand_bounds,constraints_satisfied
0,"{'A': 40.0, 'B': 0.0, 'C': 20.0}",1159.999999,"{'machine_1': 99.99999997788791, 'machine_2': ...","{'machine_1': 2.2112089936854318e-08, 'machine...","{'A': True, 'B': False, 'C': False}","{'machine_1': True, 'machine_2': True, 'demand..."


## Task 3: Scenario analysis

We now modify one parameter block at a time and compare the new plan and objective
value with the baseline solution.


In [3]:
# Store the mandatory scenarios from the practical script in one dictionary.
scenarios = {
    "Higher profit for C": {
        "profit": np.array([20, 25, 30], dtype=float),
        "capacity": capacity,
        "demand_max": demand_max,
    },
    "More capacity on machine 2": {
        "profit": profit,
        "capacity": np.array([100, 110], dtype=float),
        "demand_max": demand_max,
    },
    "Lower demand for B": {
        "profit": profit,
        "capacity": capacity,
        "demand_max": np.array([40, 10, 50], dtype=float),
    },
}

# Solve each scenario and compare it against the baseline objective value.
scenario_rows = []
for name, params in scenarios.items():
    scenario_problem, scenario_x = solve_single_period(
        params["profit"], machine_1, machine_2, params["capacity"], params["demand_max"]
    )
    scenario_rows.append(
        {
            "scenario": name,
            "plan_A": scenario_x.value[0],
            "plan_B": scenario_x.value[1],
            "plan_C": scenario_x.value[2],
            "objective_value": scenario_problem.value,
            # TODO: Compare the scenario objective with the baseline objective.
            "objective_change_vs_baseline": ...,
        }
    )

# Display the scenario comparison table.
scenario_df = pd.DataFrame(scenario_rows)
display(scenario_df)


,scenario,plan_A,plan_B,plan_C,objective_value,objective_change_vs_baseline
0,Higher profit for C,1.152749e-07,2.400000e+01,28.0,1440.0,2.800000e+02
1,More capacity on machine 2,3.000000e+01,8.400505e-08,40.0,1320.0,1.600000e+02
2,Lower demand for B,4.000000e+01,3.919002e-08,20.0,1160.0,5.390420e-07


## Task 4: Two-period extension with inventory carryover

To model period-specific demand correctly, we distinguish between production, sales, and
inventory variables. Inventory links the two periods and incurs a holding cost.


In [4]:
# Demand vectors for the two planning periods.
demand_period_1 = np.array([20, 15, 30], dtype=float)
demand_period_2 = np.array([25, 20, 35], dtype=float)
# Inventory holding cost per unit kept between the periods.
holding_cost = 1.0

# x[i, t] denotes production of product i in period t.
x = cp.Variable((3, 2), nonneg=True, name="production")
# sales[i, t] denotes units sold in period t and is limited by demand.
sales = cp.Variable((3, 2), nonneg=True, name="sales")
# inventory[i, t] stores the end-of-period stock.
inventory = cp.Variable((3, 2), nonneg=True, name="inventory")

constraints = []
for t in range(2):
    # Each period has the same machine capacities as the single-period model.
    constraints.extend(
        [machine_1 @ x[:, t] <= capacity[0], machine_2 @ x[:, t] <= capacity[1]]
    )

# Add demand bounds and inventory-balance equations.
constraints.extend(
    [
        sales[:, 0] <= demand_period_1,
        sales[:, 1] <= demand_period_2,
        # TODO: Add the inventory-balance equations for both periods.
        ...,
        inventory[:, 1] == 0,
    ]
)

# Maximize total sales profit minus the holding cost on period-1 inventory.
multi_period_problem = cp.Problem(
    cp.Maximize(cp.sum(cp.multiply(profit[:, None], sales)) - holding_cost * cp.sum(inventory[:, 0])),
    constraints,
)
multi_period_problem.solve(solver=SOLVER)

print("Two-period model status:", multi_period_problem.status)
print("Optimal objective value:", multi_period_problem.value)
display(pd.DataFrame(x.value, index=products, columns=["period_1", "period_2"]))
display(pd.DataFrame(sales.value, index=products, columns=["period_1", "period_2"]))
display(pd.DataFrame(inventory.value, index=products, columns=["end_of_period_1", "end_of_period_2"]))


Two-period model status: optimal
Optimal objective value: 2270.9999990132414


,period_1,period_2
A,20.0,25.0
B,12.0,9.0
C,24.0,23.0


,period_1,period_2
A,20.0,25.0
B,12.0,9.0
C,24.0,23.0


,end_of_period_1,end_of_period_2
A,2.442778e-08,5.407893e-15
B,4.357613e-08,6.367262e-15
C,5.888119e-08,0.000000e+00


## Task 5: Automatic experiment framework

The following helper function packages the baseline model so that many different
scenarios can be solved with minimal code duplication.


In [5]:
def production_experiment(profit, capacity, demand_max):
    # Reuse the single-period solver so the experiment function stays small and clear.
    problem, x = solve_single_period(profit, machine_1, machine_2, capacity, demand_max)
    return {"status": problem.status, "plan": x.value, "objective": problem.value}


# Generate five additional reproducible scenarios with small random perturbations.
extra_scenarios = []
for scenario_id in range(5):
    profit_trial = profit + rng.integers(-2, 6, size=3)
    capacity_trial = capacity + rng.integers(-10, 16, size=2)
    demand_trial = np.maximum(5, demand_max + rng.integers(-10, 11, size=3))
    result = production_experiment(profit_trial, capacity_trial, demand_trial)
    extra_scenarios.append(
        {
            "scenario_id": scenario_id + 1,
            "profit": profit_trial.tolist(),
            "capacity": capacity_trial.tolist(),
            "demand_max": demand_trial.tolist(),
            "objective": result["objective"],
            "plan": np.round(result["plan"], 3).tolist(),
        }
    )

# Show the experiment results in a compact summary table.
display(pd.DataFrame(extra_scenarios))


,scenario_id,profit,capacity,demand_max,objective,plan
0,1,"[19.0, 24.0, 22.0]","[102.0, 85.0]","[42.0, 34.0, 40.0]",1252.333333,"[39.667, 0.0, 22.667]"
1,2,"[21.0, 24.0, 19.0]","[114.0, 84.0]","[31.0, 31.0, 42.0]",1302.399999,"[31.0, 10.2, 21.4]"
2,3,"[24.0, 30.0, 23.0]","[106.0, 92.0]","[37.0, 23.0, 50.0]",1553.799999,"[37.0, 1.8, 26.6]"
3,4,"[21.0, 28.0, 23.0]","[97.0, 92.0]","[32.0, 27.0, 56.0]",1394.600000,"[0.0, 20.4, 35.8]"
4,5,"[19.0, 28.0, 19.0]","[103.0, 94.0]","[47.0, 37.0, 51.0]",1307.400000,"[0.0, 22.4, 35.8]"
